# Minutes of Meeting Generator — 1-Hour File on Colab (T4 GPU + pyannote)

Tuned for a **long (~1 hour) meeting** using the fastest accurate setup:
- **`large-v3`** transcription (the full, non-distilled model - the most accurate option) -> roughly 10-15 min for the hour on a T4.
- **`pyannote`** diarization (most accurate "who spoke when", runs on the GPU).
- **Google Drive** for the audio (reliable for big files; the upload button is flaky at this size).

**Before you start:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. Have your **Groq API key** (https://console.groq.com/keys) and your **Hugging Face token** ready.
   For the HF token: first accept the model terms at
   https://huggingface.co/pyannote/speaker-diarization-community-1 , then create a token with
   "Read access to contents of all public gated repos" at https://huggingface.co/settings/tokens .
3. On your PC run `python build_zip.py` to rebuild **MoM_generator.zip** — do this every time
   you change the code, or you will upload an old copy — and put your **audio file in Google
   Drive** (prefer `.mp3` / `.m4a`).

Run the cells top to bottom. Expected total for a 1-hour file: roughly **5-15 min**
(transcription is quick; pyannote diarization and the chunked LLM step are the longer parts).

## 0. Confirm the GPU is attached

In [ ]:
!nvidia-smi
# You should see a Tesla T4. If this errors, you did not select the T4 runtime (see step 1 above).

## 1. Get the code

Upload the **MoM_generator.zip** you built with `build_zip.py`.

In [ ]:
# Upload a zip of the project code.
from google.colab import files
import zipfile, os

uploaded = files.upload()                # pick MoM_generator.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

# Move into the folder that contains project.py.
for root, _dirs, fnames in os.walk('/content'):
    if 'project.py' in fnames:
        os.chdir(root)
        break
print('Working directory:', os.getcwd())

## 2. Install dependencies (core + pyannote + GPU libs)

Colab already ships PyTorch. We add faster-whisper, the pyannote diarizer, and the
**cuDNN / cuBLAS** wheels that CTranslate2 (faster-whisper's GPU engine) needs. We also
upgrade `numba`, since installing the above pulls in numpy 2.5 but Colab preinstalls
numba 0.60, which crashes (`Numba needs NumPy 2.0 or less`) as soon as pyannote lazy-imports
`librosa` later in the run.

> WARNING: installing `pyannote.audio` can occasionally pull a different torch/numpy and ask you
> to **restart the runtime**. If a later cell then says the GPU isn't visible, or you get a
> numpy/torch conflict: `Runtime -> Restart session`, then re-run from cell **3** (the cuDNN path
> fix) onward. You do NOT need to reinstall. (Other pip resolver warnings about cudf, cuml,
> pytensor, or google-colab/pandas are harmless — the pipeline doesn't use them.)

In [ ]:
!pip -q install faster-whisper groq python-docx python-dotenv scikit-learn librosa
!pip -q install pyannote.audio                        # high-accuracy diarization
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12  # CTranslate2 GPU runtime libs
!pip -q install -U numba                              # fix numba 0.60 vs numpy 2.5 crash at the pyannote step

## 3. Fix the cuDNN library path (the #1 Colab GPU gotcha)

CTranslate2 loads `libcudnn` at runtime and Colab doesn't have it on the search path by default
— the symptom is `Unable to load libcudnn_ops.so`. This points the loader at the wheels we just
installed. (`os.environ` set here is inherited by the `!python` run below.)

In [ ]:
import os, glob
nvidia_lib_dirs = sorted(set(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib')))
os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])

# Quick check that the GPU is visible to CTranslate2 itself.
import ctranslate2
print('CTranslate2 CUDA devices:', ctranslate2.get_cuda_device_count(), '(should be 1)')

## 4. Set your API key and Hugging Face token

Paste each when prompted (hidden input). `HF_TOKEN` is required for pyannote; `GROQ_API_KEY`
is required for the minutes step. Don't hard-code either in the notebook.

In [ ]:
import os, getpass
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key: ')
os.environ['HF_TOKEN']     = getpass.getpass('Hugging Face token (hf_...): ')
print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))
print('HF_TOKEN set:    ', bool(os.environ.get('HF_TOKEN')))

## 5. Point to your meeting audio (Google Drive — recommended for big files)

Mount Drive and set `AUDIO_PATH` to your 1-hour file. Prefer **`.mp3` / `.m4a`** to keep it small.
If you'd rather use the upload button, comment out the Drive lines and uncomment the upload block.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS to your file's path inside Drive:
AUDIO_PATH = '/content/drive/MyDrive/meeting_1hour.mp3'

import os
assert os.path.isfile(AUDIO_PATH), f'Not found: {AUDIO_PATH} - fix the path above.'
print('Audio:', AUDIO_PATH, f'({os.path.getsize(AUDIO_PATH)/1e6:.1f} MB)')

# --- Alternative: upload from your computer instead of Drive ---
# from google.colab import files
# audio = files.upload()
# AUDIO_PATH = next(iter(audio))
# print('Uploaded:', AUDIO_PATH)

NUM_SPEAKERS = None   # e.g. 4 if you know the count; None = pyannote auto-detects
VOCAB = ''            # names/terms the model keeps mishearing, e.g. 'Acme, KPI, Aoife'

cmd = (
    f'python project.py "{AUDIO_PATH}"'
    ' --model large-v3'
    ' --language en'      # Whisper locks onto one language from the first 30s
    ' --speakers pyannote'
    ' --save-transcript'
    ' --output output/meeting_1hr_MoM.docx'
)
if NUM_SPEAKERS:
    cmd += f' --num-speakers {NUM_SPEAKERS}'
if VOCAB:
    cmd += f' --vocab "{VOCAB}"'
print('Running:', cmd)
!{cmd}

In [ ]:
NUM_SPEAKERS = None   # e.g. 4 if you know the count; None = pyannote auto-detects

cmd = (
    f'python project.py "{AUDIO_PATH}"'
    ' --model large-v3'
    ' --speakers pyannote'
    ' --save-transcript'
    ' --output output/meeting_1hr_MoM.docx'
)
if NUM_SPEAKERS:
    cmd += f' --num-speakers {NUM_SPEAKERS}'
print('Running:', cmd)
!{cmd}

## 7. Download the results

Want real names instead of "Speaker 1/2/..."? The run above prints a sample line per detected
speaker, so re-run cell 6 with `--speaker-names "Speaker 1=Alice, Speaker 2=Bob"` added to `cmd`
(transcription reruns, but on the GPU that's quick).

In [ ]:
from google.colab import files
import os

# Transcript first: project.py writes it BEFORE the minutes step, so it is there
# even if that step fails. Downloading the .docx first would raise and skip it.
for path in ['output/meeting_1hr_MoM_transcript.txt', 'output/meeting_1hr_MoM.docx']:
    if os.path.exists(path):
        files.download(path)
    else:
        print('Not produced:', path)